# Project A: consolidation on a real corpus (M5, LoCoMo)

Registered in `docs/protocol.md` ("Registered before running: real-corpus consolidation on LoCoMo (M5)")
before any run. Does uniform replay recover more evicted facts than compaction-supervised consolidation
on natural multi-session dialogue, as it does on the synthetic corpora?

LoCoMo (CC BY-NC 4.0) is downloaded at run time and checked against the registered sha256. Probes are the
517 QA items whose answer (4 characters or more) appears verbatim in the conversation. Two folds by
conversation parity; each fold is consolidated on its own events with the other fold for validation.

Order: the compactor gate on both folds, then five seeds of `compaction` and `uniform` per fold, then the
registered paired t over the ten runs. The first (fold, seed) is timed; if the projection exceeds the
registered 15 GPU hours the notebook stops after fold 0.


In [ ]:
import glob, os, subprocess, sys, zipfile

GIT_URL = 'https://github.com/rajul-kk/context-to-weights.git'
REPO = '/kaggle/working/myrios'
MARKER = 'baselines/cascading.py'

def looks_like_source(d):
    return os.path.exists(os.path.join(d, MARKER))

if not looks_like_source(REPO):
    src = None
    for d in sorted(glob.glob('/kaggle/input/*')):
        if looks_like_source(d):
            src = d
            break
        for z in sorted(glob.glob(os.path.join(d, '*.zip'))):
            os.makedirs(REPO, exist_ok=True)
            zipfile.ZipFile(z).extractall(REPO)
            if looks_like_source(REPO):
                src = REPO
                break
        if src:
            break
    if src and src != REPO:
        subprocess.run(['cp', '-r', src, REPO], check=True)
    if not looks_like_source(REPO) and GIT_URL:
        r = subprocess.run(['git', 'clone', GIT_URL, REPO], capture_output=True, text=True)
        print(r.stdout, r.stderr)
    assert looks_like_source(REPO), (
        'No source found. Either (a) run scripts/package_source.py locally, upload the zip '
        'as a Kaggle Dataset, and attach it via Add Input, or (b) set GIT_URL above. '
        f'Searched /kaggle/input/*, saw: {sorted(glob.glob("/kaggle/input/*"))}')

os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])

try:
    import importlib.metadata as _md
    _v = _md.version('torchao')
    if tuple(int(x) for x in _v.split('.')[:2]) < (0, 16):
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'uninstall', '-y', 'torchao'])
        print(f'removed incompatible torchao {_v}')
except Exception:
    pass
exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
print('files', sorted(os.listdir('.'))[:10])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print('=' * 68)
    print('NO GPU. Kaggle installed the CPU build of torch, so this session')
    print('has no accelerator attached. Everything below will be far too slow.')
    print()
    print('Fix: right panel -> Session options -> Accelerator -> GPU T4 x2,')
    print('then Run All again. The image swaps to a CUDA torch build on restart.')
    print('=' * 68)
else:
    print('gpu', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')



## Config and data


In [ ]:
import os, sys, time, json, shutil, hashlib, urllib.request
REPO = '/kaggle/working/myrios'
os.chdir(REPO); sys.path.insert(0, REPO)
if 'run' not in dir():
    exec(open('notebooks/_runner.py').read())

LOCOMO_URL = 'https://raw.githubusercontent.com/snap-research/locomo/main/data/locomo10.json'
LOCOMO_SHA256 = '79fa87e90f04081343b8c8debecb80a9a6842b76a7aa537dc9fdf651ea698ff4'
RAW = '/kaggle/working/locomo10.json'
CFG = 'configs/kaggle_locomo.yaml'
COMPACTOR = 'Qwen/Qwen2.5-0.5B-Instruct'
COMPACTOR_ALT = 'Qwen/Qwen2.5-1.5B-Instruct'
TARGET = 'Qwen/Qwen2.5-0.5B-Instruct'
FOLDS = [0, 1]
SEEDS = [0, 1, 2, 3, 4]
BUDGET_HOURS = 15.0
OUT = '/kaggle/working/m5'
ARMS = [('ours', 'compaction'), ('uniform', 'uniform')]
os.makedirs(OUT, exist_ok=True)

if not os.path.exists(RAW):
    urllib.request.urlretrieve(LOCOMO_URL, RAW)
digest = hashlib.sha256(open(RAW, 'rb').read()).hexdigest()
assert digest == LOCOMO_SHA256, f'LoCoMo file changed upstream: {digest}'
for k in FOLDS:
    run(f'python data/load_locomo.py --input {RAW} --out artifacts/data/locomo_f{k} --verbatim-only '
        f'--min-answer-chars 4 --max-probes 100000 --n-folds 2 --fold {k}')


## Compactor gate


In [ ]:
def fold_paths(k):
    runs = f'/kaggle/working/artifacts/runs_locomo_f{k}'
    return runs, f'artifacts/data/locomo_f{k}', f'{runs}/cascading'

gate = {}
for k in FOLDS:
    runs, data, casc = fold_paths(k)
    over = f'data.dir={data} run_root={runs}'
    run(f'python baselines/cascading.py --config {CFG} --split both --out {casc} --set model.base={COMPACTOR} {over}')
    run(f'python eval/span_report.py --events {casc}/eval_events.jsonl --show 0 --out {casc}/eval_span_report.json')
    alt = f'{runs}/cascading_alt'
    run(f'python baselines/cascading.py --config {CFG} --split eval --out {alt} --set model.base={COMPACTOR_ALT} {over}')
    run(f'python eval/span_report.py --events {alt}/eval_events.jsonl --show 0 --out {alt}/eval_span_report.json')
    gate[k] = {m: json.load(open(f'{d}/eval_span_report.json')) for m, d in (('0.5B', casc), ('1.5B', alt))}
    print(f'fold {k}: ' + '  '.join(f"{m} lift {r['salience_lift']:.2f} sigma {r.get('chance_margin_sigma', float('nan')):+.2f}"
                                    for m, r in gate[k].items()), flush=True)
json.dump(gate, open(f'{OUT}/gate.json', 'w'), indent=1)
GATE_CLEARED = any(gate[k]['0.5B']['salience_lift'] > 1.0 for k in FOLDS)
print('gate cleared' if GATE_CLEARED else 'GATE NOT CLEARED: M5 stops here (registered verdict)')


## Runs


In [ ]:
def adapter_for(runs, tag):
    best = f'{runs}/sleep_{tag}/best/adapter'
    return best if os.path.exists(best) else f'{runs}/sleep_{tag}/latest/adapter'

def one_run(k, seed):
    runs, data, casc = fold_paths(k)
    report = f'{runs}/report_s{seed}'
    over = f'data.dir={data} run_root={runs} seed={seed}'
    t0 = time.time()
    for label, method in ARMS:
        run(f'python sleep/loop.py --config {CFG} --method {method} --events {casc}/eval_events.jsonl '
            f'--val-events {casc}/train_events.jsonl --run-dir {runs}/sleep_{label}_s{seed} --resume --val-limit 64 '
            f'--set model.base={TARGET} {over}')
    for label, tag in [('cascading', None)] + [(lab, f'{lab}_s{seed}') for lab, _ in ARMS]:
        a = f'--adapter {adapter_for(runs, tag)}' if tag else ''
        run(f'python eval/retention.py --config {CFG} --contexts {casc}/eval_contexts.jsonl --label {label} '
            f'--out {report}/{label} {a} --set model.base={TARGET} {over}')
    dst = f'{OUT}/fold_{k}_seed_{seed}'
    os.makedirs(dst, exist_ok=True)
    labels = ['cascading'] + [lab for lab, _ in ARMS]
    for label in labels:
        shutil.copy(f'{report}/{label}.summary.json', dst)
        rec = f'{report}/{label}.records.jsonl'
        if os.path.exists(rec):
            shutil.copy(rec, dst)
    minutes = (time.time() - t0) / 60
    json.dump({'minutes': minutes}, open(f'{dst}/times.json', 'w'))
    shutil.make_archive('/kaggle/working/m5_runs', 'zip', OUT)
    rows = {lab: json.load(open(f'{dst}/{lab}.summary.json')) for lab in labels}
    print(f'fold {k} seed {seed}: n_evicted {rows["uniform"]["n_evicted"]}  ' + '  '.join(
        f'{lab} {r["evicted_accuracy"]:.3f}' for lab, r in rows.items()) + f'  {minutes:.0f} min', flush=True)
    return minutes

if GATE_CLEARED:
    first = one_run(FOLDS[0], SEEDS[0])
    projected = first * len(FOLDS) * len(SEEDS) / 60
    print(f'projected {projected:.1f} h for {len(FOLDS) * len(SEEDS)} runs (registered limit {BUDGET_HOURS} h)')
    folds = FOLDS if projected <= BUDGET_HOURS else FOLDS[:1]
    for k in folds:
        for s in SEEDS:
            if (k, s) != (FOLDS[0], SEEDS[0]):
                one_run(k, s)
    if folds != FOLDS:
        print('stopped after fold 0 on the registered budget rule; the owner decides on fold 1')


## Registered test


In [ ]:
import glob
from scipy import stats

rows = []
for d in sorted(glob.glob(f'{OUT}/fold_*_seed_*')):
    r = {lab: json.load(open(f'{d}/{lab}.summary.json'))['evicted_accuracy'] for lab in ('cascading', 'ours', 'uniform')}
    rows.append((os.path.basename(d), r))
    print(f"{os.path.basename(d):<20} cascading {r['cascading']:.3f}  ours {r['ours']:.3f}  uniform {r['uniform']:.3f}")
if len(rows) >= 2:
    u = [r['uniform'] for _, r in rows]
    o = [r['ours'] for _, r in rows]
    c = [r['cascading'] for _, r in rows]
    t, p = stats.ttest_rel(u, o)
    tu, pu = stats.ttest_rel(u, c)
    to, po = stats.ttest_rel(o, c)
    res = {'n_runs': len(rows), 'uniform_minus_ours': float(sum(a - b for a, b in zip(u, o)) / len(u)), 't': float(t), 'p': float(p),
           'uniform_vs_floor': {'t': float(tu), 'p': float(pu)}, 'ours_vs_floor': {'t': float(to), 'p': float(po)}}
    informative = (pu < 0.05 and tu > 0) or (po < 0.05 and to > 0)
    if not informative:
        verdict = 'uninformative: neither arm beats the cascading floor'
    elif p < 0.05:
        verdict = 'replicates: uniform above compaction' if t > 0 else 'contradicted: compaction above uniform'
    else:
        verdict = 'no separation'
    res['verdict'] = verdict
    json.dump(res, open(f'{OUT}/test.json', 'w'), indent=1)
    shutil.make_archive('/kaggle/working/m5_runs', 'zip', OUT)
    print(json.dumps(res, indent=1))
